In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))
from pipeline.bronze.landing_to_bronze import add_metadata

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("base", "s3://motorsport-data-lake")
dbutils.widgets.text("target", "motorsport.bronze.telemetry")

BASE = dbutils.widgets.get("base")
TARGET = dbutils.widgets.get("target")
print(BASE)

In [0]:
raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","parquet")
    .option("cloudFiles.schemaLocation",f"{BASE}/schemas/bronze_telemetry")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load(f"{BASE}/landing/")
)

In [0]:
query = (
    add_metadata(raw)
    .writeStream
    .option("checkpointLocation",f"{BASE}/checkpoints/bronze_telemetry")
    .option("mergeSchema","true")
    .trigger(availableNow=True)
    .toTable(TARGET)
)

In [0]:
query.awaitTermination()